In [ ]:
import torch, gc

torch.cuda.empty_cache()
gc.collect()


37

In [ ]:
!nvidia-smi


Sun Feb  1 15:14:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install torch torchvision wandb thop matplotlib numpy


In [6]:
import wandb
wandb.login()


True

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader
from thop import profile


In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cuda


In [9]:
class CIFAR10DataLoader:
    def __init__(self, batch_size=64, num_workers=0):
        self.batch_size = batch_size
        self.num_workers = num_workers

        self.train_transform = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5),
                                 (0.5, 0.5, 0.5))
        ])

        self.test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5),
                                 (0.5, 0.5, 0.5))
        ])

    def get_loaders(self):
        train_set = torchvision.datasets.CIFAR10(
            root="./data",
            train=True,
            download=True,
            transform=self.train_transform
        )

        test_set = torchvision.datasets.CIFAR10(
            root="./data",
            train=False,
            download=True,
            transform=self.test_transform
        )

        train_loader = DataLoader(
            train_set,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers
        )

        test_loader = DataLoader(
            test_set,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers
        )

        return train_loader, test_loader


In [15]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


In [16]:
model = SimpleCNN().to(device)

dummy_input = torch.randn(1, 3, 32, 32).to(device)
flops, params = profile(model, inputs=(dummy_input,), verbose=False)

print(f"FLOPs: {flops/1e6:.2f} MFLOPs")
print(f"Parameters: {params/1e6:.2f} M")


FLOPs: 42.08 MFLOPs
Parameters: 2.47 M


In [17]:
def plot_grad_flow(named_parameters, epoch):
    ave_grads = []
    layers = []

    for n, p in named_parameters:
        if p.requires_grad and p.grad is not None and "bias" not in n:
            layers.append(n)
            ave_grads.append(p.grad.abs().mean().item())

    plt.figure(figsize=(12, 6))
    plt.plot(ave_grads)
    plt.xticks(range(len(layers)), layers, rotation=90)
    plt.xlabel("Layers")
    plt.ylabel("Average Gradient")
    plt.title(f"Gradient Flow - Epoch {epoch}")
    plt.tight_layout()

    wandb.log({"Gradient Flow": wandb.Image(plt)})
    plt.close()


def plot_weight_update_flow(model, prev_weights, epoch):
    update_ratios = []
    layers = []

    for name, param in model.named_parameters():
        if param.requires_grad and param.data.ndimension() > 1:
            delta_w = torch.norm(param.data - prev_weights[name])
            w_norm = torch.norm(param.data)

            update_ratios.append((delta_w / (w_norm + 1e-8)).item())
            layers.append(name)

    plt.figure(figsize=(12, 4))
    plt.plot(update_ratios, marker='o')
    plt.xticks(range(len(layers)), layers, rotation=90)
    plt.ylabel("||ΔW|| / ||W||")
    plt.title(f"Weight Update Flow - Epoch {epoch}")
    plt.tight_layout()

    wandb.log({"Weight Update Flow": wandb.Image(plt)})
    plt.close()


In [18]:
wandb.init(
    project="CIFAR10-CNN-Lab2",
    config={
        "epochs": 30,
        "batch_size": 128,
        "optimizer": "Adam",
        "loss": "CrossEntropy"
    }
)

wandb.log({
    "FLOPs (MFLOPs)": flops / 1e6,
    "Parameters (M)": params / 1e6
})


FLOPs (MFLOPs),▁
Parameters (M),▁
FLOPs (MFLOPs),42.07923
Parameters (M),2.47451


In [19]:
train_loader, test_loader = CIFAR10DataLoader().get_loaders()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 25

# ---- Initialize previous weights ONCE ----
prev_weights = {
    name: param.data.clone()
    for name, param in model.named_parameters()
    if param.requires_grad
}

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0.0

    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    # ---- Gradient flow (already existing) ----
    plot_grad_flow(model.named_parameters(), epoch)

    # ---- Weight update flow (NEW) ----
    plot_weight_update_flow(model, prev_weights, epoch)

    # ---- Update prev_weights for next epoch ----
    prev_weights = {
        name: param.data.clone()
        for name, param in model.named_parameters()
        if param.requires_grad
    }

    # ---- Log scalars ----
    wandb.log({
        "Train Loss": avg_loss,
        "Epoch": epoch
    })

    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {avg_loss:.4f}")


100%|██████████| 170M/170M [00:03<00:00, 49.2MB/s]


Epoch [1/25] - Loss: 1.6270
Epoch [2/25] - Loss: 1.2954
Epoch [3/25] - Loss: 1.1395
Epoch [4/25] - Loss: 1.0390
Epoch [5/25] - Loss: 0.9660
Epoch [6/25] - Loss: 0.9070
Epoch [7/25] - Loss: 0.8565
Epoch [8/25] - Loss: 0.8068
Epoch [9/25] - Loss: 0.7670
Epoch [10/25] - Loss: 0.7318
Epoch [11/25] - Loss: 0.6968
Epoch [12/25] - Loss: 0.6757
Epoch [13/25] - Loss: 0.6434
Epoch [14/25] - Loss: 0.6234
Epoch [15/25] - Loss: 0.6002
Epoch [16/25] - Loss: 0.5767
Epoch [17/25] - Loss: 0.5577
Epoch [18/25] - Loss: 0.5499
Epoch [19/25] - Loss: 0.5288
Epoch [20/25] - Loss: 0.5174
Epoch [21/25] - Loss: 0.4988
Epoch [22/25] - Loss: 0.4853
Epoch [23/25] - Loss: 0.4739
Epoch [24/25] - Loss: 0.4604
Epoch [25/25] - Loss: 0.4509


In [20]:
wandb.finish()


Epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
FLOPs (MFLOPs),▁
Parameters (M),▁
Train Loss,█▆▅▄▄▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
Epoch,24
FLOPs (MFLOPs),42.07923
Parameters (M),2.47451
Train Loss,0.45094
